# Manual CAT Sessions with CatEngine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/douglasrizzo/catsim/blob/main/notebooks/02_manual_cat_sessions.ipynb)

Use `CatEngine` when your software owns the CAT loop. This is the right
workflow for applications with a UI, API, or database that receives real
responses from examinees.

In [ ]:
import matplotlib

matplotlib.use("Agg")

import numpy as np

from catsim.engine import CatEngine, RunContext
from catsim.estimation import NumericalSearchEstimator
from catsim.initialization import FixedPointInitializer
from catsim.item_bank import ItemBank
from catsim.selection import MaxInfoSelector
from catsim.state import CatSessionState
from catsim.stopping import MinErrorStopper

## Create the item bank and engine

In [ ]:
item_bank = ItemBank.generate_item_bank(80, seed=11)
engine = CatEngine(
  initializer=FixedPointInitializer(0.0),
  selector=MaxInfoSelector(r_max=0.9),
  estimator=NumericalSearchEstimator(),
  stopper=MinErrorStopper(0.38, min_items=5, max_items=10),
)
context = RunContext(rng=np.random.default_rng(2024))

## Start a session

A `CatSessionState` is the runtime object you would typically persist or
serialize in an external system.

In [ ]:
session = engine.start_session(
  session_id=501,
  item_bank=item_bank,
  context=context,
  true_theta=None,
)

print(session)
print("Theta history:", session.theta_history)

## Select and apply items step by step

In a real system, the response would come from an examinee. Here we use a
small mocked response sequence to keep the example deterministic.

In [ ]:
response_stream = [True, True, False, True, False, True, True]

while not engine.should_stop(session, item_bank):
  response = response_stream[len(session.responses) % len(response_stream)]
  next_item = engine.select_next(session, item_bank, context)
  step = engine.apply_response(session, item_bank, next_item, response, context)
  print({
    "item": int(step.selected_item_id),
    "response": step.response,
    "updated_theta": round(float(step.updated_theta), 3),
    "stopped": step.stopped,
  })
  if step.stopped:
    break

## Read the session state

The application can inspect the updated state after every response.

In [ ]:
print("Administered items:", [int(item) for item in session.administered_item_ids])
print("Responses:", session.responses)
print("Theta history:", [round(float(theta), 3) for theta in session.theta_history])
print("Status:", session.status.value)
print("Stop reason:", session.stop_reason)

## Reconstruct a session

If you persist the session fields in another system, you can reconstruct a
`CatSessionState` later and continue from there.

In [ ]:
restored = CatSessionState(
  session_id=session.session_id,
  current_theta=session.latest_theta,
  true_theta=session.true_theta,
  administered_item_ids=list(session.administered_item_ids),
  responses=list(session.responses),
  theta_history=list(session.theta_history),
  status=session.status,
  stop_reason=session.stop_reason,
  metadata={"source": "example"},
)

print(restored.administered_count)
print(restored.latest_theta)

## Integration notes

In a production system, the calling application should usually persist at
least:

- `session_id`
- `current_theta` or `theta_history`
- `administered_item_ids`
- `responses`
- `status`
- any application metadata that ties the CAT session to a user or attempt

The application layer should remain responsible for authentication, UI,
scoring policies outside IRT, and storage. `catsim` should own the CAT
decision logic itself.